# System performance: latency, memory, FLOPs and energy of every method (Colab A100)

The desktop table of `system_performance.tex` for one VLM (`MODEL`), on exactly the code and conditions of its
**budget-matched accuracy run** (run that notebook first: this one reads its `config_matched.yaml` and `summary.csv`).
Per method and condition, on 20 questions sampled across the videos (one at a time, frames decoded in memory,
3 warm-ups): **TTFT** (vision side + prefill), **TPS** (64 greedy tokens), **VE / LD / overall peak dynamic memory**,
**frames encoded**, **vision-side GFLOPs**, **method overhead** (ours: the eye + head gate; baselines: their extra
work over Uniform K), **energy and power** (NVML), and the decoder's **memory vs batch size**. Details:
`system_perf/README.md`.

Results go to `MyDrive/GazeVLM/system_perf/<model>/`. Rough time on an A100: ~45 min for Qwen2.5-VL-3B
(21 conditions x 24 runs, plus the batch sweep).


In [ ]:
import os, sys, subprocess
import torch

# An A100 (bf16, 40 GB+) is needed: on a T4 (16 GB, no bf16) Uniform K's 96 frames run out of memory and
# the bf16 models crawl. Runtime > Change runtime type > A100 GPU.
gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print('GPU:', f'{gpu.name}, {gpu.total_memory / 1e9:.0f} GB' if gpu else 'none')
assert gpu and torch.cuda.get_device_capability(0)[0] >= 8 and gpu.total_memory > 35e9, \
    'needs an A100 GPU runtime (bf16, >= 40 GB): Runtime > Change runtime type > A100'

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
# num2words: needed by SmolVLM2's processor (install before transformers is first imported)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest', 'num2words', 'pyyaml', 'nvidia-ml-py'],
               check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
try:                                         # Hugging Face token from the Colab secret HF_TOKEN
    from google.colab import userdata
    from huggingface_hub import login
    if userdata.get('HF_TOKEN'):
        login(token=userdata.get('HF_TOKEN'))
except Exception as ex:
    print('no HF_TOKEN secret:', type(ex).__name__)

MODEL = 'qwen2.5'            # qwen2.5 | smolvlm | internvl (the models with a budget-matched accuracy run)
LABEL = {'qwen2.5': 'Qwen2.5-VL-3B', 'smolvlm': 'SmolVLM2-2.2B', 'internvl': 'InternVL3.5-4B'}[MODEL]
ACC = '/content/drive/MyDrive/GazeVLM/accuracy_tables/' + {'qwen2.5': 'qwen25_matched', 'smolvlm': 'smolvlm_matched',
                                                            'internvl': 'internvl_matched'}[MODEL]
MATCHED = f'{ACC}/config_matched.yaml'
SYS = '/content/drive/MyDrive/GazeVLM/system_perf'
QUESTIONS, BATCH = 20, '1,2,4,8,16,32'
assert os.path.exists(MATCHED), f'run the budget-matched accuracy notebook for {MODEL} first ({MATCHED} missing)'
print('conditions from', MATCHED, '-> results in', f'{SYS}/{MODEL}')


In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_system_perf.py', 'tests/test_accuracy_tables.py', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed'


## Measure

Parity first, then every row and condition of the matched config on the sampled questions. Each line printed is one
question: TTFT (vision side), TPS, VE / LD peak memory, frames encoded, visual tokens. Rerunning appends; the report
keeps the last record per question.


In [ ]:
get_ipython().system(f'python -m system_perf.measure --config {MATCHED} --model {MODEL} --questions {QUESTIONS} '
                     f'--out-dir {SYS} --batch-sizes {BATCH}')


## Report

Mean +- s.d. per condition, speed-ups and memory savings vs Uniform K, method overheads, accuracy from the accuracy
run; `system_table.csv` and `system_table.tex` (rows for the desktop table of `system_performance.tex`).


In [ ]:
get_ipython().system(f'python -m system_perf.report --results {SYS} --model {MODEL} --accuracy {ACC}/summary.csv '
                     f'--label "{LABEL}"')
print(open(f'{SYS}/{MODEL}/system_table.tex').read())
